In [1]:
# Gemini gives me this witchcraft to help install the DPU master package

import sys

sys.path.append('/home/xilinx/jupyter_notebooks/FASTAR/DPU_wysong_test/DPU-PYNQ-master')

import pynq_dpu
from pynq_dpu import DpuOverlay

Couldn't import vart, check if library installed and is on path.


ImportError: libpython3.9.so.1.0: cannot open shared object file: No such file or directory

In [ ]:
import numpy as np
from pynq_dpu import DpuOverlay

print("1. Loading Overlay & Model...")
overlay = DpuOverlay("DPU.bit")
overlay.load_model("resnet50_b800.xmodel")

# Grab the DPU runner instance
dpu = overlay.runner

# Ask the model what shapes it expects
inputTensors = dpu.get_input_tensors()
outputTensors = dpu.get_output_tensors()

shapeIn = tuple(inputTensors[0].dims)
shapeOut = tuple(outputTensors[0].dims)
print(f"Model expects input shape: {shapeIn}")
print(f"Model yields output shape: {shapeOut}")

print("\n2. Generating Random Static...")
# Create empty arrays formatted for C-style memory layout
input_data = [np.empty(shapeIn, dtype=np.float32, order="C")]
output_data = [np.empty(shapeOut, dtype=np.float32, order="C")]

# Fill the input array with random noise (dummy image)
input_data[0][:] = np.random.rand(*shapeIn)

print("\n3. Firing up the DPU...")
# Push the data to the hardware!
job_id = dpu.execute_async(input_data, output_data)
dpu.wait(job_id)

print("\n4. DONE!")
# ResNet50 outputs 1000 numbers (probabilities for 1000 classes). 
# We find the index of the highest number.
prediction_id = np.argmax(output_data[0])
print(f"SUCCESS! The DPU processed the image and classified the random noise as Class ID: {prediction_id}")